# Auto MPG Deep Learning Assignment
Implementing a regression model with PyTorch, ONNX, and Auto MPG data based on class starter code.

In [ ]:
import numpy as np
import torch
import pandas as pd
import random
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
from mlxtend.plotting import heatmap
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import r2_score

### Parameters

In [ ]:
batch_size    = 16
learning_rate = 0.005
N_Epochs      = 100
epsilon       = 0.0001

### Read Data (Auto MPG)

In [ ]:
# Using a direct URL for the Auto MPG dataset
url = "http://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data"
column_names = ['MPG', 'Cylinders', 'Displacement', 'Horsepower', 'Weight',
                'Acceleration', 'Model Year', 'Origin']

# Read the data, handling the whitespace delimiter and missing '?' values
raw_dataset = pd.read_csv(url, names=column_names,
                          na_values='?', comment='\t',
                          sep=' ', skipinitialspace=True)

# Drop missing values (Auto MPG has 6 missing horsepower values)
dataset = raw_dataset.dropna()

dataset.head()

### Process Data

In [ ]:
# Features (X) and Target (y - MPG)
X = dataset.drop('MPG', axis=1).values
y = dataset['MPG'].values.reshape(-1, 1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fix data types for PyTorch
X_train = X_train.astype(np.float32)
X_test  = X_test.astype(np.float32)
y_train = y_train.astype(np.float32)
y_test  = y_test.astype(np.float32)

X_train_tr = torch.from_numpy(X_train)
X_test_tr  = torch.from_numpy(X_test)
y_train_tr = torch.from_numpy(y_train)
y_test_tr  = torch.from_numpy(y_test)

# Normalization
x_means      = X_train_tr.mean(0, keepdim=True) 
x_deviations = X_train_tr.std(0, keepdim=True) + epsilon

train_ds = TensorDataset(X_train_tr, y_train_tr)
train_dl = DataLoader(train_ds, batch_size, shuffle=True)

### Neural Network Architecture

In [ ]:
class DL_Net(nn.Module):
    def __init__(self, x_means, x_deviations, input_size):
        super().__init__()
        self.x_means      = x_means
        self.x_deviations = x_deviations
        
        self.linear1 = nn.Linear(input_size, 10)
        self.act1    = nn.ReLU()
        self.linear2 = nn.Linear(10, 6)
        self.act2    = nn.ReLU()
        self.linear3 = nn.Linear(6, 1)
        
    def forward(self, x):
        x = (x - self.x_means) / self.x_deviations
        x = self.linear1(x)
        x = self.act1(x)
        x = self.linear2(x)
        x = self.act2(x)
        y_pred = self.linear3(x)
        return y_pred

### Training Loop

In [ ]:
def training_loop(N_Epochs, model, loss_fn, opt):
    for epoch in range(N_Epochs):
        for xb, yb in train_dl:
            y_pred = model(xb)
            loss   = loss_fn(y_pred, yb)
            
            opt.zero_grad()
            loss.backward()
            opt.step()
            
        if epoch % 20 == 0:
            print(f"Epoch {epoch}, loss={loss.item():.4f}")

input_size = X_train.shape[1]
model = DL_Net(x_means, x_deviations, input_size)
opt     = torch.optim.Adam(model.parameters(), lr=learning_rate)
loss_fn = F.mse_loss

training_loop(N_Epochs, model, loss_fn, opt)

### Evaluate and Export to ONNX

In [ ]:
model.eval()
with torch.no_grad():
    y_pred_test = model(X_test_tr)
    print("Testing R^2: ", r2_score(y_test_tr.numpy(), y_pred_test.numpy()))

# Export to ONNX
dummy_input = torch.randn(1, input_size)
torch.onnx.export(model, dummy_input, "model.onnx", 
                  input_names=['input'], output_names=['output'],
                  dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}})
print("Model exported to model.onnx")